In [1]:
import numpy as np
import pandas as pd

# Circumferential Cracks

## 1. Gamma Exponent Model (GEM)

[GEM](https://www.sciencedirect.com/science/article/pii/S2667143321000470?via%3Dihub) a fracture mechanics methodology used to predict the failure pressure of pipeline flaws, including circumferential surface cracks. It is unique because it is mathematically focused on flaw depth rather than length, making it highly effective for assessing surface flaws

### Step-by-Step Guide to the GEM Assessment
To assess a circumferential surface crack using the GEM, follow these steps:

**Step 1: Determine Material Properties**

You must first define the pipeline's material characteristics:

* **Flow Stress ($S_{flow}$):** This is defined as the material's yield strength plus a constant: $S_{flow} = \text{Yield Strength} + 69 \text{ MPa}$ (or $+10 \text{ ksi}$).
* **Young’s Modulus ($E$):** The material's modulus of elasticity.
* **Poisson’s Ratio ($\nu$):** Typically assumed to be 0.3 for steel.

**Step 2: Calculate Fracture Toughness ($K_c$)**

The GEM requires a "plane strain" toughness value, often estimated from Charpy V-notch (CVN) impact test data. For surface cracks, the modified Hahn correlation is typically used:

$$K_c^2 = \frac{CVN}{Area}(1-2\nu)^2 \frac{E}{(1-\nu^2)}$$

*Note: "Area" refers to the cross-sectional area at the machined notch of the Charpy test sample.*

Axial cracks through wall would use:

$$ K_c^2 = \frac{0.53 CVN^{1.28}}{(1-2\nu)^2}E$$

**Step 3: Calculate the Gamma Exponent ($\gamma$)**

The gamma exponent balances the influence of the flaw's dimensions relative to the pipe diameter ($D$):

$$\gamma = \min\left(\frac{4c}{D}, 1\right)$$

Where $c$ is the half-length of the crack. For circumferential cracks, the limit for $\gamma$ is 1.0.

**Step 4: Predict the Failure Strength ($S_{pred}$)**

Use the core GEM equation to calculate the predicted failure strength:

$$\frac{S_{pred}}{S_{flow}} = \left(1 - \frac{a}{t}\right)^\gamma \cdot \frac{2}{\pi} \arccos \left[ \exp \left( -\frac{K_c^2}{10a S_{flow}^2} \right) \right]$$

* $a$: Flaw depth.

* $t$: Pipe wall thickness.

* $a/t$: Ratio of flaw depth to wall thickness.

**Step 5: Apply Correction Factors**

For circumferential cracks specifically, a correction factor is applied to the predicted strength to account for lateral bending of the pipe.

**The GEM Mathematical Structure Explained**

The model consists of two distinct components that "balance" the failure process:

1. Flow Strength Term: $\left(1 - \frac{a}{t}\right)^\gamma$ 
    
    This represents the remaining ligament of the pipe. It is analogous to the "Maxey" equation used in older models like ASME B31G but uses the gamma exponent instead of the Folias factor.

2. Fracture Toughness Term: $\frac{2}{\pi} \arccos \left[ \exp \left( -\frac{K_c^2}{10a S_{flow}^2} \right) \right]$

    This accounts for the material's resistance to crack propagation. A key innovation here is that the term is based on crack depth ($a$) rather than half-length ($c$), which is standard in older "log-secant" models.

In [4]:
import math

def assess_circumferential_crack_gem(yield_strength, E, nu, CVN, Area, D, t, a, c):
    """
    Evaluates the failure strength of a pipeline with a circumferential surface crack 
    using the Gamma Exponent Model (GEM).
    
    IMPORTANT: Ensure consistent units are used. 
    Standard Metric Recommendation:
    - yield_strength: MPa
    - E (Young's Modulus): MPa
    - nu (Poisson's ratio): Unitless (typically ~0.3 for steel)
    - CVN (Charpy V-Notch impact energy): Joules (J)
    - Area (Charpy sample cross-sectional area): mm^2
    - D (Pipe diameter): mm
    - t (Pipe wall thickness): mm
    - a (Flaw depth): mm
    - c (Flaw half-length): mm
    
    Returns:
    - S_bend: The predicted failure strength under lateral bending (MPa).
    """
    
    # =========================================================================
    # STEP 1: Determine Material Properties (Flow Stress)
    # =========================================================================
    # Flow stress (S_flow) represents the stress level at which the material 
    # continues to deform plastically. In the GEM model (metric), it is defined 
    # as the Yield Strength plus 69 MPa (which is approx. 10 ksi).
    S_flow = yield_strength + 69.0
    
    
    # =========================================================================
    # STEP 2: Calculate Fracture Toughness (Kc^2)
    # =========================================================================
    # Because surface cracks operate under plane strain conditions, we use a 
    # modified Hahn correlation to estimate the square of the fracture toughness 
    # (Kc^2) directly from Charpy V-notch (CVN) impact data.
    # The Poisson's ratio (nu) terms act as a penalty to account for the triaxial 
    # stress state that hinders yielding at the crack tip.
    Kc_sq = (CVN / Area) * ((1 - 2 * nu)**2) * (E / (1 - nu**2))
    
    
    # =========================================================================
    # STEP 3: Calculate the Gamma Exponent (gamma)
    # =========================================================================
    # The gamma exponent balances the flaw's dimensions relative to the pipe.
    # For crack-like flaws (including circumferential cracks), it is defined 
    # as 4 times the flaw half-length (c) divided by the Pipe Diameter (D), 
    # but strictly capped at a maximum limit of 1.0.
    gamma = min(4.0 * c / D, 1.0)
    
    
    # =========================================================================
    # STEP 4: Predict the Base Failure Strength (S_pred)
    # =========================================================================
    # The GEM equation consists of two distinct balancing terms:
    
    # Term A: The Flow Strength Term (Remaining Ligament)
    # This accounts for the remaining healthy material in the pipe wall.
    ligament_term = (1.0 - (a / t))**gamma
    
    # Term B: The Fracture Toughness Term
    # This accounts for the material's resistance to the crack growing deeper.
    # Note: Unlike older models, the denominator in the exponent uses 10 * a 
    # (flaw depth) rather than flaw length.
    exponent_val = -Kc_sq / (10.0 * a * (S_flow**2))
    toughness_term = (2.0 / math.pi) * math.acos(math.exp(exponent_val))
    
    # Base predicted failure strength (under pure internal pressure)
    S_pred_base = S_flow * ligament_term * toughness_term
    
    
    # =========================================================================
    # STEP 5: Apply Correction Factor for Circumferential Cracks
    # =========================================================================
    # The base GEM assumes internal pressure. Because circumferential cracks 
    # are heavily influenced by the lateral bending of the pipeline, an empirical 
    # correction factor based on the flaw-depth-to-wall-thickness ratio (a/t) 
    # is applied.
    bending_correction_factor = (0.25 * (a / t)) + 1.0
    
    # Final predicted failure strength under bending
    S_bend = S_pred_base * bending_correction_factor
    
    
    # Return the final calculated failure strength
    return S_bend

In [5]:
# =============================================================================
# EXAMPLE USAGE:
# =============================================================================
if __name__ == "__main__":
    # Sample Input Data (Metric)
    inputs = {
        "yield_strength": 450.0, # MPa (approx. API 5L X65)
        "E": 207000.0,           # MPa (Standard Young's Modulus for steel)
        "nu": 0.3,               # Poisson's ratio for steel
        "CVN": 45.0,             # Joules
        "Area": 80.0,            # mm^2 (Standard full-size Charpy specimen area)
        "D": 508.0,              # mm (20-inch pipe)
        "t": 10.0,               # mm
        "a": 3.0,                # mm (crack depth is 30% of wall thickness)
        "c": 50.0                # mm (crack total length is 100mm, so half-length c=50)
    }
    
    # Run the model
    predicted_strength = assess_circumferential_crack_gem(**inputs)
    
    print(f"Predicted Failure Strength (S_bend): {predicted_strength:.2f} MPa")

Predicted Failure Strength (S_bend): 21.96 MPa


## 2. API579/FFS-1 | BS7910


Both the API 579/ASME FFS-1 and BS 7910 methodologies evaluate crack-like flaws using a **Failure Assessment Diagram (FAD)**. The FAD approach is highly effective because it simultaneously evaluates a pipeline's susceptibility to two distinct failure modes: brittle fracture and plastic collapse.

Below is a step-by-step mathematical guide to performing a "Level 2" FAD assessment for a cylinder (pipeline) containing a surface crack based on these standards.